In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from tensorflow.keras.models import load_model

backbone = load_model("/content/drive/MyDrive/Backbone_model_V-Zeus.keras.keras")

print("Zeus backbone loaded")

Zeus backbone loaded


In [32]:
for layer in backbone.layers:
    layer.trainable = False

In [3]:
from tensorflow.keras.models import load_model

stage2_model = load_model("/content/drive/MyDrive/classifier_model_V_Hera.keras")

In [46]:
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model

x = backbone.output

x = Dense(512, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

x = Dense(256, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Dense(128, activation="relu")(x)

type_output = Dense(3, activation="softmax")(x)

hera_model = Model(backbone.input, type_output, name="classifier_model_V_Hera")

In [47]:
from tensorflow.keras.optimizers import Adam

hera_model.compile(
    optimizer=Adam(learning_rate=2e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [48]:
TRAIN_DIR = "/content/drive/MyDrive/disaster_final/train"
VAL_DIR = "/content/drive/MyDrive/disaster_final/validation"
TEST_DIR = "/content/drive/MyDrive/disaster_final/test"

In [49]:
DISASTER_TYPES = ["earthquake","flood","wildfire"]
SEVERITY_LEVELS = ["high","low","medium"]

TYPE_LABEL = {
    "earthquake":0,
    "flood":1,
    "wildfire":2
}

In [50]:
import os
import numpy as np

def load_paths(dataset_dir):

    paths=[]
    labels=[]

    for disaster in DISASTER_TYPES:
        for severity in SEVERITY_LEVELS:

            folder=os.path.join(dataset_dir,disaster,severity)

            if not os.path.exists(folder):
                continue

            for file in os.listdir(folder):

                if file.lower().endswith((".jpg",".jpeg",".png")):

                    paths.append(os.path.join(folder,file))
                    labels.append(TYPE_LABEL[disaster])

    return np.array(paths),np.array(labels)

In [38]:
train_paths,train_labels = load_paths(TRAIN_DIR)
val_paths,val_labels = load_paths(VAL_DIR)

In [39]:
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 16

def load_image(path,label):

    img=tf.io.read_file(path)
    img=tf.io.decode_image(img,channels=3)

    img.set_shape([None,None,3])

    img=tf.image.resize(img,[IMG_SIZE,IMG_SIZE])

    # augmentation
    img=tf.image.random_flip_left_right(img)
    img=tf.image.random_brightness(img,0.2)
    img=tf.image.random_contrast(img,0.8,1.2)

    img=tf.cast(img,tf.float32)/255.0

    return img,label

In [40]:
train_ds = tf.data.Dataset.from_tensor_slices((train_paths,train_labels))

train_ds = train_ds.map(load_image)\
                   .shuffle(2000)\
                   .batch(BATCH_SIZE)\
                   .prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths,val_labels))

val_ds = val_ds.map(load_image)\
               .batch(BATCH_SIZE)

In [51]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True
)

hera_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[early_stop]
)

Epoch 1/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 97s 316ms/step - accuracy: 0.7516 - loss: 0.6115 - val_accuracy: 0.9036 - val_loss: 0.3417
Epoch 2/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 49s 149ms/step - accuracy: 0.9071 - loss: 0.2776 - val_accuracy: 0.9114 - val_loss: 0.2398
Epoch 3/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 49s 147ms/step - accuracy: 0.9109 - loss: 0.2440 - val_accuracy: 0.9271 - val_loss: 0.1944
Epoch 4/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 48s 147ms/step - accuracy: 0.9157 - loss: 0.2330 - val_accuracy: 0.9361 - val_loss: 0.1763
Epoch 5/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 55s 193ms/step - accuracy: 0.9215 - loss: 0.2315 - val_accuracy: 0.9462 - val_loss: 0.1527
Epoch 6/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 77s 146ms/step - accuracy: 0.9229 - loss: 0.2079 - val_accuracy: 0.9395 - val_loss: 0.1605
Epoch 7/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 79s 148ms/step - accuracy: 0.9193 - loss: 0.2187 - val_accuracy: 0.9294 - val_loss: 0.1724
Epoch 8/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 49s 150ms/step - accuracy: 0.9303 - loss: 0

In [52]:
hera_model.save("/content/drive/MyDrive/classifier_model_V_Hera_fixed2.0.keras")

print("Hera model saved to Google Drive!")

Hera model saved to Google Drive!


In [53]:
TEST_DIR = "/content/drive/MyDrive/disaster_final/test"

test_paths, test_labels = load_paths(TEST_DIR)

print("Test images:", len(test_paths))

Test images: 462


In [54]:
test_ds = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))

test_ds = test_ds.map(load_image)\
                 .batch(BATCH_SIZE)

In [55]:
hera_model.evaluate(test_ds)

29/29 ━━━━━━━━━━━━━━━━━━━━ 16s 536ms/step - accuracy: 0.9090 - loss: 0.2990


[0.3659420609474182, 0.8831169009208679]